In [1]:
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(os.path.join('..', '.env'))

FRED_API_KEY = os.getenv('FRED_API_KEY')
print(f"FRED API Key: is {'set' if FRED_API_KEY else 'not set'}")

FRED API Key: is set


In [2]:
START_DATE = "2015-10-08"  #XLRE Introduced in October 2015 & FED rate didnt change till DEC 2015 for 7 years straight
END_DATE = datetime.today().strftime('%Y-%m-%d')

print(f"Data will be fetched from {START_DATE} to {END_DATE}")

Data will be fetched from 2015-10-08 to 2026-04-26


In [3]:
FRED_SERIES = [
    #FED's anchor rate
    ('DFF', 'Effective Federal Funds Rate'), # Daily Federal Funds Rate

    #FED's target range
    ('DFEDTARU', 'Fed Target Range Upper Limit'), # Daily
    ('DFEDTARL', 'Fed Target Range Lower Limit'), # Daily

    # -- Downstreams -- Housing
    ('MORTGAGE30US', '30-Year Fixed Rate Mortgage Average in the United States'), # Weekly (every thursday)
    ('DGS10', '10-Year Treasury Yield'), # Daily

    # -- Downstreams -- Consumer Credit
    ('TERMCBCCALLNS', 'Commercial Bank Interest Rate on Credit Card Plans'), # Monthly

    # -- Downstreams -- currency
    ('DTWEXBGS', 'Trade Weighted U.S. Dollar Index'), # Daily (This indicator covers most number of currencies than DXY)

    # -- Downstreams -- Equity
    ('SP500', 'S&P 500 Index'), # Daily

    # Macro Context (why the FED is doing what it is doing?)
    ('CPIAUCSL', 'CPI for All Urban Consumers: All Items in U.S. City Average'), # Monthly
    ('GDPC1', 'Real Gross Domestic Product'), # Quarterly ('GDP' isn't adjusted to inflation, but GDPC1 is)
    ('UNRATE', 'Unemployment Rate'), # Monthly

    # -- Optional Enrichments --
    ('VIXCLS', 'CBOE Volatility Index'), # Daily (This is a measure of market volatility, often referred to as the "fear index.")
    ('DPRIME', 'Prime Rate') # Daily (The prime rate is the interest rate that commercial banks charge their most creditworthy customers, often used as a benchmark for various loans and credit products.)
]

SECTOR_ETFS = {
    'XLF': 'Financials',
    'XLK': 'Technology',
    'XLE': 'Energy',
    'XLV': 'Health Care',
    'XLY': 'Consumer Discretionary',
    'XLRE': 'Real Estate',
}

print(f'{len(FRED_SERIES)} FRED series will be fetched')
print(f'{len(SECTOR_ETFS)} sector ETFs will be fetched')

13 FRED series will be fetched
6 sector ETFs will be fetched


In [4]:
RAW_DATA_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DATA_DIR = os.path.join('..', 'data', 'processed')

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"Raw data will be stored in: {os.path.abspath(RAW_DATA_DIR)}")
print(f"Processed data will be stored in: {os.path.abspath(PROCESSED_DATA_DIR)}")

Raw data will be stored in: /Users/malithj/Desktop/Masters/Spring 26/DV/Final Project/FollowTheFed/data/raw
Processed data will be stored in: /Users/malithj/Desktop/Masters/Spring 26/DV/Final Project/FollowTheFed/data/processed


In [5]:
import pandas as pd
from fredapi import Fred

fred_client = Fred(FRED_API_KEY)

def fetch_one(seriesid, start = START_DATE, end = END_DATE):
    """
    Download a single series from FRED and return it as a clean dataframe.

    Args:
        seriesid (str): The FRED series ID to fetch.
        start (str): The start date for the data in 'YYYY-MM-DD' format.
        end (str): The end date for the data in 'YYYY-MM-DD' format.

    Returns:
        A DataFrame, not a pandas series, with columns: ['date', 'value']
    """

    raw = fred_client.get_series(
        seriesid,
        observation_start=start,
        observation_end=end
    )

    df = raw.reset_index()
    df.columns = ['date', seriesid.lower()]

    df['date'] = pd.to_datetime(df['date'])

    df[seriesid.lower()] = pd.to_numeric(df[seriesid.lower()], errors='coerce')

    return df

In [6]:
# from fredapi import Fred
# fred_client = Fred(FRED_API_KEY)
# raw_test = fred_client.get_series('DFF', observation_start=START_DATE, observation_end=END_DATE)
# print(raw_test.head())
# print(type(raw_test))
# print(raw_test.columns)

In [7]:
fred_data = {}

total = len(FRED_SERIES)

for i, (seriesid, description) in enumerate(FRED_SERIES):
    try:
        df = fetch_one(seriesid)
        fred_data[seriesid] = df
        
        rows = df.iloc[:, 1].notna().sum()

        print(f'✔️ [{i+1:>2}/{total}] {seriesid:<16s} | {rows:>6,} rows | {description}')

    except Exception as e:
        print(f'❌ [{i+1:>2}/{total}] {seriesid:<16s} | Error: {str(e)}')

print()
print(f'Downloaded: {len(fred_data)}/{total} series successfully.')


✔️ [ 1/13] DFF              |  3,851 rows | Effective Federal Funds Rate
✔️ [ 2/13] DFEDTARU         |  3,853 rows | Fed Target Range Upper Limit
✔️ [ 3/13] DFEDTARL         |  3,853 rows | Fed Target Range Lower Limit
✔️ [ 4/13] MORTGAGE30US     |    551 rows | 30-Year Fixed Rate Mortgage Average in the United States
✔️ [ 5/13] DGS10            |  2,634 rows | 10-Year Treasury Yield
✔️ [ 6/13] TERMCBCCALLNS    |     42 rows | Commercial Bank Interest Rate on Credit Card Plans
✔️ [ 7/13] DTWEXBGS         |  2,623 rows | Trade Weighted U.S. Dollar Index
✔️ [ 8/13] SP500            |  2,515 rows | S&P 500 Index
✔️ [ 9/13] CPIAUCSL         |    125 rows | CPI for All Urban Consumers: All Items in U.S. City Average
✔️ [10/13] GDPC1            |     41 rows | Real Gross Domestic Product
✔️ [11/13] UNRATE           |    125 rows | Unemployment Rate
✔️ [12/13] VIXCLS           |  2,679 rows | CBOE Volatility Index
✔️ [13/13] DPRIME           |  2,667 rows | Prime Rate

Downloaded: 13/13 serie

In [8]:
for series_id, name in FRED_SERIES:
    df = fred_data[series_id]
    
    total_rows = len(df)
    valid_rows = df.iloc[:, 1].notnull().sum()

    #Getting the days between two successive observations (Both date and value should be present for it to be considered valid)
    valid_rows_df = df.dropna()
    gaps = valid_rows_df['date'].diff().dt.days
    typical_gap = gaps.mode()[0]  # Most common gap in days

    max_gap = gaps.max()

    if typical_gap <= 1:
        freq = 'daily'
    elif typical_gap <= 7:
        freq = 'weekly'
    elif typical_gap <= 32:
        freq = 'monthly'
    elif typical_gap <= 95:
        freq = 'quarterly'
    else:
        freq = f'every {typical_gap} days'

    print(
        f'{series_id:<16s} | '
        f'{valid_rows:>5,} pts | '
        f'gap: {typical_gap:.0f}d (typical) {max_gap:.0f}d (max) | '
        f'{freq}'
    )

DFF              | 3,851 pts | gap: 1d (typical) 1d (max) | daily
DFEDTARU         | 3,853 pts | gap: 1d (typical) 1d (max) | daily
DFEDTARL         | 3,853 pts | gap: 1d (typical) 1d (max) | daily
MORTGAGE30US     |   551 pts | gap: 7d (typical) 8d (max) | weekly
DGS10            | 2,634 pts | gap: 1d (typical) 4d (max) | daily
TERMCBCCALLNS    |    42 pts | gap: 92d (typical) 92d (max) | quarterly
DTWEXBGS         | 2,623 pts | gap: 1d (typical) 6d (max) | daily
SP500            | 2,515 pts | gap: 1d (typical) 4d (max) | daily
CPIAUCSL         |   125 pts | gap: 31d (typical) 61d (max) | monthly
GDPC1            |    41 pts | gap: 92d (typical) 92d (max) | quarterly
UNRATE           |   125 pts | gap: 31d (typical) 61d (max) | monthly
VIXCLS           | 2,679 pts | gap: 1d (typical) 4d (max) | daily
DPRIME           | 2,667 pts | gap: 1d (typical) 4d (max) | daily


- 61day max gap for CPI and Unemployment rate: FEDs didn't publish it because of the 43-day gov shutdown.
- TERMCBCCALLNS - Rates have published every 3 months even if the FRED says its monthly

In [9]:
for series_id, df in fred_data.items():
    path = os.path.join(RAW_DATA_DIR, f'{series_id}.csv')
    df.to_csv(path, index=False)

files = sorted(os.listdir(RAW_DATA_DIR))
print(f"Saved {len(files)} files to {RAW_DATA_DIR}:")
print()
for f in files:
    if f.endswith('.csv'):
        size_kb = os.path.getsize(os.path.join(RAW_DATA_DIR, f)) / 1024
        print(f" - {f:<25s} {size_kb:>7.1f} KB")

Saved 16 files to ../data/raw:

 - CPIAUCSL.csv                  2.3 KB
 - DFEDTARL.csv                 58.4 KB
 - DFEDTARU.csv                 58.3 KB
 - DFF.csv                      59.8 KB
 - DGS10.csv                    42.3 KB
 - DPRIME.csv                   41.3 KB
 - DTWEXBGS.csv                 52.4 KB
 - GDPC1.csv                     0.8 KB
 - MORTGAGE30US.csv              8.6 KB
 - SP500.csv                    47.6 KB
 - TERMCBCCALLNS.csv             1.7 KB
 - UNRATE.csv                    1.9 KB
 - VIXCLS.csv                   45.0 KB
 - fomc_decisions.csv            2.3 KB
 - sector_etfs.csv             645.9 KB


In [10]:
import yfinance as yf

tickers = list(SECTOR_ETFS.keys())
print(f"Fetching data for {len(tickers)} sector ETFs: {', '.join(tickers)}")

raw_etf_data = yf.download(
    tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True
)

print(f'Shape of downloaded ETF data: {raw_etf_data.shape}')
print(f'Columns in downloaded ETF data: {raw_etf_data.columns.names}')
print(f'Start date: {raw_etf_data.index.min().date()}')
print(f'End date: {raw_etf_data.index.max().date()}')


Fetching data for 6 sector ETFs: XLF, XLK, XLE, XLV, XLY, XLRE


[*********************100%***********************]  6 of 6 completed

Shape of downloaded ETF data: (2651, 30)
Columns in downloaded ETF data: ['Price', 'Ticker']
Start date: 2015-10-08
End date: 2026-04-24


In [11]:
raw_etf_data['Close'][tickers].head()

Ticker,XLF,XLK,XLE,XLV,XLY,XLRE
Date,,,,,,
2015-10-08,15.593830,18.335552,22.757893,57.202827,34.641205,21.044127
2015-10-09,15.493829,18.415464,22.610306,57.463108,34.681370,21.009300
2015-10-12,15.507159,18.437664,22.318409,57.614223,34.846481,21.141651
2015-10-13,15.387157,18.397705,22.085550,56.892166,34.659046,21.009300
2015-10-14,15.260485,18.362185,22.275776,56.783020,34.306526,21.009300


In [12]:
close = raw_etf_data['Close'][tickers].copy()

close.columns = [f'{t.lower()}_close' for t in tickers]

returns = close.pct_change()

returns.columns = [c.replace('_close', '_return') for c in close.columns]

sector_df = pd.concat([close, returns], axis=1).reset_index()

sector_df.rename(columns={'Date': 'date'}, inplace=True)

sector_df['date'] = pd.to_datetime(sector_df['date']).dt.tz_localize(None)

print(f'Shape: {sector_df.shape}')
print(f'Columns: {list(sector_df.columns)}')
print()

for t, name in SECTOR_ETFS.items():
    col = f'{t.lower()}_close'
    valid = sector_df[col].notna().sum()
    latest = sector_df[sector_df[col].notna()][col].iloc[-1]
    print(f'  ✓ {t:<5s} ({name:<24s}) | {valid:>5,} days | latest: ${latest:,.2f}')

print()
print(sector_df.head(3).to_string(index=False))

Shape: (2651, 13)
Columns: ['date', 'xlf_close', 'xlk_close', 'xle_close', 'xlv_close', 'xly_close', 'xlre_close', 'xlf_return', 'xlk_return', 'xle_return', 'xlv_return', 'xly_return', 'xlre_return']

  ✓ XLF   (Financials              ) | 2,651 days | latest: $51.42
  ✓ XLK   (Technology              ) | 2,651 days | latest: $160.22
  ✓ XLE   (Energy                  ) | 2,651 days | latest: $56.87
  ✓ XLV   (Health Care             ) | 2,651 days | latest: $144.18
  ✓ XLY   (Consumer Discretionary  ) | 2,651 days | latest: $118.69
  ✓ XLRE  (Real Estate             ) | 2,651 days | latest: $43.83

      date  xlf_close  xlk_close  xle_close  xlv_close  xly_close  xlre_close  xlf_return  xlk_return  xle_return  xlv_return  xly_return  xlre_return
2015-10-08  15.593830  18.335552  22.757893  57.202827  34.641205   21.044127         NaN         NaN         NaN         NaN         NaN          NaN
2015-10-09  15.493829  18.415464  22.610306  57.463108  34.681370   21.009300   -0.006413  

In [13]:
path = os.path.join(RAW_DATA_DIR, 'sector_etfs.csv')
sector_df.to_csv(path, index=False)

print(f"Saved sector ETF data to {path} ({os.path.getsize(path) / 1024:.1f} KB)")

Saved sector ETF data to ../data/raw/sector_etfs.csv (646.1 KB)


In [14]:
import io

csv_text = """date,action,change_bps,upper_target,lower_target
2015-10-28,hold,0,0.25,0.00
2015-12-16,hike,25,0.50,0.25
2016-01-27,hold,0,0.50,0.25
2016-03-16,hold,0,0.50,0.25
2016-04-27,hold,0,0.50,0.25
2016-06-15,hold,0,0.50,0.25
2016-07-27,hold,0,0.50,0.25
2016-09-21,hold,0,0.50,0.25
2016-11-02,hold,0,0.50,0.25
2016-12-14,hike,25,0.75,0.50
2017-02-01,hold,0,0.75,0.50
2017-03-15,hike,25,1.00,0.75
2017-05-03,hold,0,1.00,0.75
2017-06-14,hike,25,1.25,1.00
2017-07-26,hold,0,1.25,1.00
2017-09-20,hold,0,1.25,1.00
2017-11-01,hold,0,1.25,1.00
2017-12-13,hike,25,1.50,1.25
2018-01-31,hold,0,1.50,1.25
2018-03-21,hike,25,1.75,1.50
2018-05-02,hold,0,1.75,1.50
2018-06-13,hike,25,2.00,1.75
2018-08-01,hold,0,2.00,1.75
2018-09-26,hike,25,2.25,2.00
2018-11-08,hold,0,2.25,2.00
2018-12-19,hike,25,2.50,2.25
2019-01-30,hold,0,2.50,2.25
2019-03-20,hold,0,2.50,2.25
2019-05-01,hold,0,2.50,2.25
2019-06-19,hold,0,2.50,2.25
2019-07-31,cut,-25,2.25,2.00
2019-09-18,cut,-25,2.00,1.75
2019-10-30,cut,-25,1.75,1.50
2019-12-11,hold,0,1.75,1.50
2020-01-29,hold,0,1.75,1.50
2020-03-03,cut,-50,1.25,1.00
2020-03-15,cut,-100,0.25,0.00
2020-04-29,hold,0,0.25,0.00
2020-06-10,hold,0,0.25,0.00
2020-07-29,hold,0,0.25,0.00
2020-09-16,hold,0,0.25,0.00
2020-11-05,hold,0,0.25,0.00
2020-12-16,hold,0,0.25,0.00
2021-01-27,hold,0,0.25,0.00
2021-03-17,hold,0,0.25,0.00
2021-04-28,hold,0,0.25,0.00
2021-06-16,hold,0,0.25,0.00
2021-07-28,hold,0,0.25,0.00
2021-09-22,hold,0,0.25,0.00
2021-11-03,hold,0,0.25,0.00
2021-12-15,hold,0,0.25,0.00
2022-01-26,hold,0,0.25,0.00
2022-03-16,hike,25,0.50,0.25
2022-05-04,hike,50,1.00,0.75
2022-06-15,hike,75,1.75,1.50
2022-07-27,hike,75,2.50,2.25
2022-09-21,hike,75,3.25,3.00
2022-11-02,hike,75,4.00,3.75
2022-12-14,hike,50,4.50,4.25
2023-02-01,hike,25,4.75,4.50
2023-03-22,hike,25,5.00,4.75
2023-05-03,hike,25,5.25,5.00
2023-06-14,hold,0,5.25,5.00
2023-07-26,hike,25,5.50,5.25
2023-09-20,hold,0,5.50,5.25
2023-11-01,hold,0,5.50,5.25
2023-12-13,hold,0,5.50,5.25
2024-01-31,hold,0,5.50,5.25
2024-03-20,hold,0,5.50,5.25
2024-05-01,hold,0,5.50,5.25
2024-06-12,hold,0,5.50,5.25
2024-07-31,hold,0,5.50,5.25
2024-09-18,cut,-50,5.00,4.75
2024-11-07,cut,-25,4.75,4.50
2024-12-18,cut,-25,4.50,4.25
2025-01-29,hold,0,4.50,4.25
2025-03-19,hold,0,4.50,4.25
2025-05-07,hold,0,4.50,4.25
2025-06-18,hold,0,4.50,4.25
2025-07-30,hold,0,4.50,4.25
2025-09-17,cut,-25,4.25,4.00
2025-10-29,cut,-25,4.00,3.75
2025-12-10,cut,-25,3.75,3.50
2026-01-28,hold,0,3.75,3.50
2026-03-18,hold,0,3.75,3.50"""

fomc_df = pd.read_csv(io.StringIO(csv_text))
fomc_df['date'] = pd.to_datetime(fomc_df['date'])

# ── Verify ──

action_counts = fomc_df['action'].value_counts()

print(f'Total FOMC meetings: {len(fomc_df)}')
print(f'  Hikes: {action_counts.get("hike", 0)}')
print(f'  Cuts:  {action_counts.get("cut", 0)}')
print(f'  Holds: {action_counts.get("hold", 0)}')
print(f'  Range: {fomc_df["date"].min().strftime("%b %Y")} → {fomc_df["date"].max().strftime("%b %Y")}')
print(f'  Current rate: {fomc_df.iloc[-1]["lower_target"]}% – {fomc_df.iloc[-1]["upper_target"]}%')
print()

# Show all
print('All FOMC meetings:')
print(fomc_df.to_string(index=False))

Total FOMC meetings: 85
  Hikes: 20
  Cuts:  11
  Holds: 54
  Range: Oct 2015 → Mar 2026
  Current rate: 3.5% – 3.75%

All FOMC meetings:
      date action  change_bps  upper_target  lower_target
2015-10-28   hold           0          0.25          0.00
2015-12-16   hike          25          0.50          0.25
2016-01-27   hold           0          0.50          0.25
2016-03-16   hold           0          0.50          0.25
2016-04-27   hold           0          0.50          0.25
2016-06-15   hold           0          0.50          0.25
2016-07-27   hold           0          0.50          0.25
2016-09-21   hold           0          0.50          0.25
2016-11-02   hold           0          0.50          0.25
2016-12-14   hike          25          0.75          0.50
2017-02-01   hold           0          0.75          0.50
2017-03-15   hike          25          1.00          0.75
2017-05-03   hold           0          1.00          0.75
2017-06-14   hike          25          1.25       

In [15]:
path = os.path.join(RAW_DATA_DIR, 'fomc_decisions.csv')
fomc_df.to_csv(path, index=False)

print(f"Saved FOMC decisions to {path} ({os.path.getsize(path) / 1024:.1f} KB)")

Saved FOMC decisions to ../data/raw/fomc_decisions.csv (2.3 KB)


In [16]:
date_spine = pd.DataFrame({'date': pd.date_range(start=START_DATE, end=END_DATE, freq='D')})

print(f'Step 1: Date spine created — {len(date_spine):,} days')

Step 1: Date spine created — 3,854 days


In [17]:
master_df = date_spine.copy()

for series_id, df in fred_data.items():
    master_df = master_df.merge(df, on='date', how='left')


print(f'Step 2: Merged {len(fred_data)} FRED series')
print(master_df.head(10).to_string(index=False))

Step 2: Merged 13 FRED series
      date  dff  dfedtaru  dfedtarl  mortgage30us  dgs10  termcbccallns  dtwexbgs  sp500  cpiaucsl  gdpc1  unrate  vixcls  dprime
2015-10-08 0.13      0.25       0.0          3.76   2.12            NaN  109.6452    NaN       NaN    NaN     NaN   17.42    3.25
2015-10-09 0.13      0.25       0.0           NaN   2.12            NaN  109.0782    NaN       NaN    NaN     NaN   17.08    3.25
2015-10-10 0.13      0.25       0.0           NaN    NaN            NaN       NaN    NaN       NaN    NaN     NaN     NaN     NaN
2015-10-11 0.13      0.25       0.0           NaN    NaN            NaN       NaN    NaN       NaN    NaN     NaN     NaN     NaN
2015-10-12 0.13      0.25       0.0           NaN    NaN            NaN       NaN    NaN       NaN    NaN     NaN   16.17     NaN
2015-10-13 0.13      0.25       0.0           NaN   2.06            NaN  109.4119    NaN       NaN    NaN     NaN   17.67    3.25
2015-10-14 0.13      0.25       0.0           NaN   1.99    

In [18]:
# Forward Fill Method

data_cols = [c for c in master_df.columns if c!= 'date']
master_df[data_cols] = master_df[data_cols].ffill()

In [19]:
# Count how much coverage we have after filling
filled_pct = master_df[data_cols].notna().mean() * 100
print(filled_pct)

dff              100.000000
dfedtaru         100.000000
dfedtarl         100.000000
mortgage30us     100.000000
dgs10            100.000000
termcbccallns     99.377270
dtwexbgs         100.000000
sp500             94.810586
cpiaucsl          99.377270
gdpc1             97.794499
unrate            99.377270
vixcls           100.000000
dprime           100.000000
dtype: float64


In [20]:
# Merging Sector ETFs
master_df = master_df.merge(sector_df, on='date', how='left')

sector_cols = [c for c in master_df.columns if c.endswith('_close')]
master_df[sector_cols] = master_df[sector_cols].ffill()

# Adding 0s for the weekends returns only columns (since the price is not changing on weekends, the return is 0% except for the START DATE
return_cols = [c for c in master_df.columns if c.endswith('_return')]
master_df[return_cols] = master_df[return_cols].fillna(0).where(master_df['date'] > pd.to_datetime(START_DATE))

print(f'Step 3: Merged sector ETFs and forward-filled {len(sector_cols)} columns')
print(master_df.head(10).to_string(index=False))

Step 3: Merged sector ETFs and forward-filled 6 columns
      date  dff  dfedtaru  dfedtarl  mortgage30us  dgs10  termcbccallns  dtwexbgs  sp500  cpiaucsl  gdpc1  unrate  vixcls  dprime  xlf_close  xlk_close  xle_close  xlv_close  xly_close  xlre_close  xlf_return  xlk_return  xle_return  xlv_return  xly_return  xlre_return
2015-10-08 0.13      0.25       0.0          3.76   2.12            NaN  109.6452    NaN       NaN    NaN     NaN   17.42    3.25  15.593830  18.335552  22.757893  57.202827  34.641205   21.044127         NaN         NaN         NaN         NaN         NaN          NaN
2015-10-09 0.13      0.25       0.0          3.76   2.12            NaN  109.0782    NaN       NaN    NaN     NaN   17.08    3.25  15.493829  18.415464  22.610306  57.463108  34.681370   21.009300   -0.006413    0.004358   -0.006485    0.004550    0.001159    -0.001655
2015-10-10 0.13      0.25       0.0          3.76   2.12            NaN  109.0782    NaN       NaN    NaN     NaN   17.08    3.25  15.

In [ ]:
fomc_merge = fomc_df.copy()
fomc_merge.columns = [
    'date',
    'fomc_action',
    'fomc_change_bps',
    'fomc_upper',
    'fomc_lower'
]

master_df = master_df.merge(fomc_merge, on='date', how='left')


master_df['is_fomc_meeting'] = master_df['fomc_action'].notna().astype(int)
fomc_count = master_df['is_fomc_meeting'].sum()
print(fomc_count)

date             100.000000
dff              100.000000
dfedtaru         100.000000
dfedtarl         100.000000
mortgage30us     100.000000
dgs10            100.000000
termcbccallns     99.377270
dtwexbgs         100.000000
sp500             94.810586
cpiaucsl          99.377270
gdpc1             97.794499
unrate            99.377270
vixcls           100.000000
dprime           100.000000
xlf_close        100.000000
xlk_close        100.000000
xle_close        100.000000
xlv_close        100.000000
xly_close        100.000000
xlre_close       100.000000
xlf_return        99.974053
xlk_return        99.974053
xle_return        99.974053
xlv_return        99.974053
xly_return        99.974053
xlre_return       99.974053
dtype: float64


In [22]:
# ── Final summary ──
print()
print(f'MASTER TABLE: {master_df.shape[0]:,} rows  {master_df.shape[1]} columns')
print(f'Date range: {master_df["date"].min().strftime("%Y-%m-%d")} → {master_df["date"].max().strftime("%Y-%m-%d")}')
print()
print('Column coverage after forward-fill:')
for col in master_df.columns:
    n = master_df[col].notna().sum()
    pct = n / len(master_df) * 100
    # Build a visual bar: █ for filled, ░ for empty
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    print(f'  {col:<22s} {bar} {pct:>5.1f}%')


MASTER TABLE: 3,854 rows  26 columns
Date range: 2015-10-08 → 2026-04-26

Column coverage after forward-fill:
  date                   ████████████████████ 100.0%
  dff                    ████████████████████ 100.0%
  dfedtaru               ████████████████████ 100.0%
  dfedtarl               ████████████████████ 100.0%
  mortgage30us           ████████████████████ 100.0%
  dgs10                  ████████████████████ 100.0%
  termcbccallns          ███████████████████░  99.4%
  dtwexbgs               ████████████████████ 100.0%
  sp500                  ██████████████████░░  94.8%
  cpiaucsl               ███████████████████░  99.4%
  gdpc1                  ███████████████████░  97.8%
  unrate                 ███████████████████░  99.4%
  vixcls                 ████████████████████ 100.0%
  dprime                 ████████████████████ 100.0%
  xlf_close              ████████████████████ 100.0%
  xlk_close              ████████████████████ 100.0%
  xle_close              ████████████████